In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import BinaryLabelDatasetMetric, ClassificationMetric

# load adult (same way as your backend)
cols = ["age","workclass","fnlwgt","education","education-num","marital-status",
        "occupation","relationship","race","sex","capital-gain","capital-loss",
        "hours-per-week","native-country","income"]

df = pd.read_csv("/Users/sameekshakumar/aif_env/lib/python3.13/site-packages/aif360/data/raw/adult/adult.data",
                 header=None, names=cols, skipinitialspace=True, na_values="?").dropna()

df["income"] = (df["income"].str.strip() == ">50K").astype(float)
df["sex"]    = (df["sex"] == "Male").astype(float)
from sklearn.preprocessing import LabelEncoder
for c in ["workclass","education","marital-status","occupation","relationship","native-country","race"]:
    df[c] = LabelEncoder().fit_transform(df[c].astype(str))

def run_experiment(df, poison_pct=0.0):
    df_p = df.copy()
    # flip labels for female records
    female_idx = df_p[df_p["sex"] == 0].sample(frac=poison_pct, random_state=42).index
    df_p.loc[female_idx, "income"] = 1 - df_p.loc[female_idx, "income"]
    
    dataset = BinaryLabelDataset(df=df_p, label_names=["income"],
                                  protected_attribute_names=["sex"],
                                  favorable_label=1.0, unfavorable_label=0.0,
                                  privileged_protected_attributes=[[1.0]],
                                  unprivileged_protected_attributes=[[0.0]])
    
    np.random.seed(1)
    train, test = dataset.split([0.7], shuffle=True)
    
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(train.features)
    X_te = scaler.transform(test.features)
    
    lmod = LogisticRegression(max_iter=500)
    lmod.fit(X_tr, train.labels.ravel())
    
    pred = test.copy(deepcopy=True)
    pred.labels = lmod.predict(X_te).reshape(-1,1)
    
    priv   = [{"sex": 1}]
    unpriv = [{"sex": 0}]
    cm = ClassificationMetric(test, pred, unprivileged_groups=unpriv, privileged_groups=priv)
    dm = BinaryLabelDatasetMetric(test, unprivileged_groups=unpriv, privileged_groups=priv)
    
    return {
        "poison_pct": poison_pct,
        "accuracy": round(0.5*(cm.true_positive_rate()+cm.true_negative_rate()), 4),
        "disparate_impact": round(cm.disparate_impact(), 4),
        "stat_parity_diff": round(cm.statistical_parity_difference(), 4),
    }

results = [run_experiment(df, p) for p in [0.0, 0.02, 0.05, 0.10, 0.15, 0.20]]
for r in results:
    print(r)

/opt/anaconda3/envs/aif360/lib/python3.10/site-packages/inFairness/utils/ndcg.py:37: FutureWarning: We've integrated functorch into PyTorch. As the final step of the integration, `functorch.vmap` is deprecated as of PyTorch 2.0 and will be deleted in a future version of PyTorch >= 2.3. Please use `torch.vmap` instead; see the PyTorch 2.0 release notes and/or the `torch.func` migration guide for more details https://pytorch.org/docs/main/func.migrating.html
  vect_normalized_discounted_cumulative_gain = vmap(
/opt/anaconda3/envs/aif360/lib/python3.10/site-packages/inFairness/utils/ndcg.py:48: FutureWarning: We've integrated functorch into PyTorch. As the final step of the integration, `functorch.vmap` is deprecated as of PyTorch 2.0 and will be deleted in a future version of PyTorch >= 2.3. Please use `torch.vmap` instead; see the PyTorch 2.0 release notes and/or the `torch.func` migration guide for more details https://pytorch.org/docs/main/func.migrating.html
  monte_carlo_vect_ndcg =

{'poison_pct': 0.0, 'accuracy': 0.6961, 'disparate_impact': 0.1823, 'stat_parity_diff': -0.1738}
{'poison_pct': 0.02, 'accuracy': 0.688, 'disparate_impact': 0.2009, 'stat_parity_diff': -0.1662}
{'poison_pct': 0.05, 'accuracy': 0.6788, 'disparate_impact': 0.2604, 'stat_parity_diff': -0.1484}
{'poison_pct': 0.1, 'accuracy': 0.6636, 'disparate_impact': 0.3266, 'stat_parity_diff': -0.1285}
{'poison_pct': 0.15, 'accuracy': 0.652, 'disparate_impact': 0.4443, 'stat_parity_diff': -0.1016}
{'poison_pct': 0.2, 'accuracy': 0.6405, 'disparate_impact': 0.6037, 'stat_parity_diff': -0.0677}


In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

# use all features EXCEPT sex to predict sex
X = df.drop(columns=["sex", "income"])
y = df["sex"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

lmod = LogisticRegression(max_iter=500)
scores = cross_val_score(lmod, X_scaled, y, cv=5, scoring="accuracy")
print(f"Sex prediction accuracy from non-protected features: {scores.mean():.4f} (+/- {scores.std():.4f})")

Sex prediction accuracy from non-protected features: 0.7731 (+/- 0.0054)


In [5]:
# reload df fresh for race experiment
cols = ["age","workclass","fnlwgt","education","education-num","marital-status",
        "occupation","relationship","race","sex","capital-gain","capital-loss",
        "hours-per-week","native-country","income"]

df_raw = pd.read_csv("/Users/sameekshakumar/aif_env/lib/python3.13/site-packages/aif360/data/raw/adult/adult.data",
                 header=None, names=cols, skipinitialspace=True, na_values="?").dropna()

df_race = df_raw.copy()
df_race["income"] = (df_race["income"].str.strip() == ">50K").astype(float)
df_race["sex"]    = (df_race["sex"].str.strip() == "Male").astype(float)
df_race["race"]   = (df_race["race"].str.strip() == "White").astype(float)
for c in ["workclass","education","marital-status","occupation","relationship","native-country"]:
    df_race[c] = LabelEncoder().fit_transform(df_race[c].astype(str))

def run_experiment_race(df, poison_pct=0.0):
    df_p = df.copy()
    nonwhite_idx = df_p[df_p["race"] == 0].sample(frac=poison_pct, random_state=42).index
    df_p.loc[nonwhite_idx, "income"] = 1 - df_p.loc[nonwhite_idx, "income"]
    
    dataset = BinaryLabelDataset(df=df_p, label_names=["income"],
                                  protected_attribute_names=["race"],
                                  favorable_label=1.0, unfavorable_label=0.0,
                                  privileged_protected_attributes=[[1.0]],
                                  unprivileged_protected_attributes=[[0.0]])
    np.random.seed(1)
    train, test = dataset.split([0.7], shuffle=True)
    
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(train.features)
    X_te = scaler.transform(test.features)
    
    lmod = LogisticRegression(max_iter=500)
    lmod.fit(X_tr, train.labels.ravel())
    
    pred = test.copy(deepcopy=True)
    pred.labels = lmod.predict(X_te).reshape(-1,1)
    
    priv   = [{"race": 1}]
    unpriv = [{"race": 0}]
    cm = ClassificationMetric(test, pred, unprivileged_groups=unpriv, privileged_groups=priv)
    
    return {
        "poison_pct": poison_pct,
        "accuracy": round(0.5*(cm.true_positive_rate()+cm.true_negative_rate()), 4),
        "disparate_impact": round(cm.disparate_impact(), 4),
        "stat_parity_diff": round(cm.statistical_parity_difference(), 4),
    }

results_race = [run_experiment_race(df_race, p) for p in [0.0, 0.02, 0.05, 0.10, 0.15, 0.20]]
for r in results_race:
    print(r)

{'poison_pct': 0.0, 'accuracy': 0.697, 'disparate_impact': 0.4404, 'stat_parity_diff': -0.0949}
{'poison_pct': 0.02, 'accuracy': 0.6922, 'disparate_impact': 0.4995, 'stat_parity_diff': -0.0834}
{'poison_pct': 0.05, 'accuracy': 0.6861, 'disparate_impact': 0.6128, 'stat_parity_diff': -0.0629}
{'poison_pct': 0.1, 'accuracy': 0.6805, 'disparate_impact': 0.8384, 'stat_parity_diff': -0.0255}
{'poison_pct': 0.15, 'accuracy': 0.675, 'disparate_impact': 1.0044, 'stat_parity_diff': 0.0007}
{'poison_pct': 0.2, 'accuracy': 0.6709, 'disparate_impact': 1.2312, 'stat_parity_diff': 0.0346}
